In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler

import re


In [ ]:
df = pd.read_csv('/content/avatar_comments_trailer1_full.csv')
df.head()


,text
0,I NEED to go see this!!! The 1st 2 movies are ...
1,I can&#39;t wait. I just hope it&#39;s as good...
2,🙌🙌💯💯🙌💯🌎🌎🌎🌍🌍🌍🌏🌏🌏✔✔✔🖥🖥🖥
3,I can&#39;t tell if Neytiri is either talking ...
4,"this reminds me of genshin impact in some way,..."


In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    # 3. Remove HTML tags & entities
    text = BeautifulSoup(text, "html.parser").get_text()
    # 4. Remove non-English characters (non ASCII)
    text = text.encode("ascii", "ignore").decode()
    # 5. Clean symbols, numbers, and punctuation
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text) # Remove punctuation
    text = re.sub(r"\d+", "", text) # Remove numbers
    text = re.sub(r"[^a-z\s]", "", text) # Keep only letters and spaces
    text = re.sub(r"\s+", " ", text).strip()
    # 6. Remove stopwords and 7. Lemmatization
    text = " ".join([lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words])
    return text

# Apply preprocessing to the 'text' column and create 'clean_text'
df['clean_text'] = df['text'].apply(preprocess_text)

# 2. Drop duplicates and missing values based on 'clean_text'
df.drop_duplicates(inplace=True)
df.dropna(subset=['clean_text'], inplace=True)

# 8. Filter out short texts
df = df[df['clean_text'].str.count(' ') >= 2]

df.head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,text,clean_text
0,I NEED to go see this!!! The 1st 2 movies are ...,need go see st movie definitely among favorite...
1,I can&#39;t wait. I just hope it&#39;s as good...,cant wait hope good first two took year get do...
3,I can&#39;t tell if Neytiri is either talking ...,cant tell neytiri either talking jake spider last
4,"this reminds me of genshin impact in some way,...",reminds genshin impact way cannot wait watch b...
6,The only thing that interests me is the music....,thing interest music dont care rest


In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    score = sia.polarity_scores(text)['compound']
    if score > 0.05:
        return 'positive'
    elif score < -0.05:
        return 'negative'
    else:
        return 'neutral'

df['sentiment'] = df['clean_text'].apply(get_sentiment)
df['sentiment'].value_counts()


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


,count
sentiment,
positive,3939
neutral,3015
negative,2763


In [5]:

df = pd.read_csv('/content/avatar_comments_trailer_clean.csv',
                 sep=None, engine='python', on_bad_lines='skip')

# Bersihkan kolom
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df.columns = df.columns.str.replace(';', '').str.replace(',', '').str.strip()

if 'sentiment' in df.columns:
    df['sentiment'] = df['sentiment'].astype(str).str.replace(';', '').str.strip()

# Aplikasikan pembersihan teks yang lebih baik
df['processed_text'] = df['text'].apply(advanced_text_cleaning)

# Hapus teks kosong atau terlalu pendek
df = df[df['processed_text'].str.len() > 5].reset_index(drop=True)

print(f"Dataset size after cleaning: {len(df)}")
print(f"Sentiment distribution:\n{df['sentiment'].value_counts()}")

Dataset size after cleaning: 12716
Sentiment distribution:
sentiment
positive    5139
neutral     3914
negative    3663
Name: count, dtype: int64


In [6]:
# ========================================
# 3. BALANCED SAMPLING
# ========================================
rus = RandomUnderSampler(sampling_strategy='auto', random_state=42)
X_res, y_res = rus.fit_resample(df[['processed_text']], df['sentiment'])
print(f"\nDistribusi setelah undersampling:\n{y_res.value_counts()}")


Distribusi setelah undersampling:
sentiment
negative    3663
neutral     3663
positive    3663
Name: count, dtype: int64


In [7]:
# 4. LABEL ENCODING
# ========================================
le = LabelEncoder()
y_encoded = le.fit_transform(y_res)
print(f"\nLabel mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")


Label mapping: {'negative': np.int64(0), 'neutral': np.int64(1), 'positive': np.int64(2)}


In [8]:
# 5. TF-IDF DENGAN PARAMETER OPTIMAL
# ========================================
# Strategi 1: TF-IDF dengan ngram yang lebih luas dan parameter seimbang
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 4),
    sublinear_tf=True,
    min_df=2,
    max_df=0.85,
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'\w{1,}',
    use_idf=True,
    smooth_idf=True,
    norm='l2'
)

X_tfidf = tfidf.fit_transform(X_res['processed_text'])
print(f"\nTF-IDF shape: {X_tfidf.shape}")



TF-IDF shape: (10989, 8000)


In [9]:
# ========================================
# 6. SPLIT DATA
# ========================================
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

#Menggunakan Model Logistic Regression dengan GridSearchCV

In [48]:

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score


lr_base = LogisticRegression(
    max_iter=5000,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    tol=0.0001
)


param_grid = {
    'C': [0.1, 0.01, 0.005, 1.0],
    'penalty': ['l1', 'l2'],
    'solver': ['saga'],
}


grid_search = GridSearchCV(
    estimator=lr_base,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)


grid_search.fit(X_train, y_train)


lr_best_model = grid_search.best_estimator_


y_train_pred_gs = lr_best_model.predict(X_train)
y_test_pred_gs = lr_best_model.predict(X_test)

train_acc_gs = accuracy_score(y_train, y_train_pred_gs)
test_acc_gs = accuracy_score(y_test, y_test_pred_gs)

print(f"✓ Akurasi Train (Model Terbaik): {train_acc_gs*100:.2f}%")
print(f"✓ Akurasi Test (Model Terbaik) : {test_acc_gs*100:.2f}%")


Fitting 5 folds for each of 8 candidates, totalling 40 fits
✓ Akurasi Train (Model Terbaik): 86.34%
✓ Akurasi Test (Model Terbaik) : 86.21%


INFERENSI MODEL LOGISTIC REGRESSION


In [52]:
def predict_sentiment(texts):
    if isinstance(texts, str):
        texts = [texts]
    X_new = tfidf.transform(texts)
    preds = lr_best_model.predict(X_new)
    for text, label in zip(texts, preds):
        print(f"Text: {text}\nSentiment: {label}\n")

test_texts = [
    "I really love this product, it's amazing!",
    "The service was terrible and disappointing."
]

predict_sentiment(test_texts)


Text: I really love this product, it's amazing!
Sentiment: positive

Text: The service was terrible and disappointing.
Sentiment: negative



#Menggunakan Model LightGBM

In [61]:
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report

model_lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=32,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model_lgbm.fit(X_train, y_train)

y_train_pred = model_lgbm.predict(X_train)
y_test_pred = model_lgbm.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"✓ Akurasi Train: {train_acc*100:.2f}%")
print(f"✓ Akurasi Test : {test_acc*100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(y_test, y_test_pred))



LIGHTGBM MODEL (RINGAN + AKURAT)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.083706 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 34422
[LightGBM] [Info] Number of data points in the train set: 8791, number of used features: 1261
[LightGBM] [Info] Start training from score -1.098385
[LightGBM] [Info] Start training from score -1.098726
[LightGBM] [Info] Start training from score -1.098726
✓ Akurasi Train: 94.74%
✓ Akurasi Test : 83.58%

Classification Report:

              precision    recall  f1-score   support

    negative       0.84      0.79      0.82       732
     neutral       0.81      0.89      0.85       733
    positive       0.86      0.82      0.84       733

    accuracy                           0.84      2198
   macro avg       0.84      0.84      0.84      2198
weighted avg       0.84      0.84      0.84      2

In [64]:
#INFERENSI MODEL
def predict_sentiment(texts):
    X_new = tfidf.transform(texts)
    preds = model_lgbm.predict(X_new)
    for t, p in zip(texts, preds):
        print(f"{t} → {p}")

test_texts = [
    "this movie was amazing and touching",
    "the plot was boring and predictable",
    "it was okay"
]

predict_sentiment(test_texts)


this movie was amazing and touching → positive
the plot was boring and predictable → negative
it was okay → neutral


In [65]:
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


df = pd.read_csv('/content/avatar_comments_trailer_clean.csv')
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df.columns = df.columns.str.strip()

texts = df['text'].astype(str).tolist()
labels = df['sentiment'].astype(str)


In [66]:
# Encode label
le = LabelEncoder()
y = le.fit_transform(labels)

# Tokenisasi teks
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

# Konversi ke urutan angka
X = tokenizer.texts_to_sequences(texts)
X = pad_sequences(X, maxlen=100)  # potong/panjang jadi 100 token


In [68]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)


In [69]:
model = Sequential([
    Embedding(input_dim=10000, output_dim=64, input_length=100),
    GRU(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(len(le.classes_), activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


In [70]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=6,
    batch_size=64,
    verbose=1
)


Epoch 1/6
140/140 ━━━━━━━━━━━━━━━━━━━━ 26s 152ms/step - accuracy: 0.4874 - loss: 0.9997 - val_accuracy: 0.7248 - val_loss: 0.6383
Epoch 2/6
140/140 ━━━━━━━━━━━━━━━━━━━━ 19s 139ms/step - accuracy: 0.8008 - loss: 0.5179 - val_accuracy: 0.8024 - val_loss: 0.5032
Epoch 3/6
140/140 ━━━━━━━━━━━━━━━━━━━━ 23s 158ms/step - accuracy: 0.8966 - loss: 0.3109 - val_accuracy: 0.8173 - val_loss: 0.4788
Epoch 4/6
140/140 ━━━━━━━━━━━━━━━━━━━━ 20s 145ms/step - accuracy: 0.9343 - loss: 0.2153 - val_accuracy: 0.8231 - val_loss: 0.5112
Epoch 5/6
140/140 ━━━━━━━━━━━━━━━━━━━━ 20s 142ms/step - accuracy: 0.9509 - loss: 0.1667 - val_accuracy: 0.8318 - val_loss: 0.5218
Epoch 6/6
140/140 ━━━━━━━━━━━━━━━━━━━━ 21s 150ms/step - accuracy: 0.9591 - loss: 0.1378 - val_accuracy: 0.8239 - val_loss: 0.5890


In [71]:
# EVALUASI MODEL
y_train_pred = np.argmax(model.predict(X_train), axis=1)
y_test_pred = np.argmax(model.predict(X_test), axis=1)

# Hitung akurasi
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

# Tampilkan hasil

print("EVALUASI MODEL GRU")
print(f"✓ Akurasi Train : {train_acc*100:.2f}%")
print(f"✓ Akurasi Test  : {test_acc*100:.2f}%")


279/279 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step
120/120 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step
EVALUASI MODEL GRU
✓ Akurasi Train : 97.53%
✓ Akurasi Test  : 82.39%


In [72]:
#INFERENSI MODEL
def predict_sentiment(texts):
    seq = tokenizer.texts_to_sequences(texts)
    pad = pad_sequences(seq, maxlen=100)
    preds = np.argmax(model.predict(pad), axis=1)
    for t, p in zip(texts, preds):
        print(f"{t} → {le.inverse_transform([p])[0]}")

# Contoh pengujian
test_texts = [
    "this movie is absolutely wonderful",
    "it was boring and too long",
    "just an average film"
]
predict_sentiment(test_texts)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
this movie is absolutely wonderful → positive
it was boring and too long → negative
just an average film → neutral


In [73]:
 pip freeze requirements.txt

absl-py==1.4.0
absolufy-imports==0.3.1
accelerate==1.11.0
aiofiles==24.1.0
aiohappyeyeballs==2.6.1
aiohttp==3.13.1
aiosignal==1.4.0
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.11.2
alembic==1.17.0
altair==5.5.0
annotated-doc==0.0.3
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.11.0
anywidget==0.9.18
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
array_record==0.8.1
arrow==1.4.0
arviz==0.22.0
astropy==7.1.1
astropy-iers-data==0.2025.10.20.0.39.8
astunparse==1.6.3
atpublic==5.1
attrs==25.4.0
audioread==3.0.1
Authlib==1.6.5
autograd==1.8.0
babel==2.17.0
backcall==0.2.0
beartype==0.22.3
beautifulsoup4==4.13.5
betterproto==2.0.0b6
bigframes==2.26.0
bigquery-magics==0.10.3
bleach==6.2.0
blinker==1.9.0
blis==1.3.0
blobfile==3.1.0
blosc2==3.10.2
bokeh==3.7.3
Bottleneck==1.4.2
bqplot==0.12.45
branca==0.8.2
Brotli==1.1.0
build==1.3.0
CacheControl==0.14.3
cachetools==5.5.2
catalogue==2.0.10
certifi==2025.10.5
cffi==2.0.0
chardet==5.2.0
charset-normalizer==3